In [372]:

import torch
import numpy as np
from numpy.dtypes import StringDType
import csv
from random import randrange
import subprocess
import mmap

In [373]:
def countLines (file_path) -> int:
  line_count = 0
  with open(file_path, "r") as python_filehandle:
    with mmap.mmap(python_filehandle.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_filehandle:
        while mmap_filehandle.readline():
            line_count += 1

  return line_count

In [374]:
def retrieveSentence (sentence_idx, tokens_list, sentence_offsets) -> str:
    end_idx = len(tokens_list) if sentence_idx+1 == len(sentence_offsets) else sentence_offsets[sentence_idx+1]
    print(end_idx)
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_list[sentence_offsets[sentence_idx]:end_idx]).strip()

In [375]:
def retrieveSubtext(subtext_idx, tokens_list, subtext_offsets) -> str:
    end_idx = len(tokens_list) if subtext_idx+1 == len(subtext_offsets) else subtext_offsets[subtext_idx+1]
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_list[subtext_offsets[subtext_idx]:end_idx]).strip()

In [376]:
def retrieveSubtextBeginning(subtext_idx, tokens_list, subtext_offsets) -> str:
    end_idx = subtext_offsets[subtext_idx] + 15
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_list[subtext_offsets[subtext_idx]:end_idx]).strip()

In [377]:
def stringifyTokensTensor(tokens_tensor) -> str:
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_tensor.tolist()).strip()

In [378]:
token_vocab_length = countLines("bpe_token_indices.csv")

In [379]:
bpe_token_indices_file = open("bpe_token_indices.csv", "r")
tokenised_chu_words_training_file = open("tokenised_chu_words_training_deepcleaned.csv", "r")

In [380]:
tokens_list = []
word_offsets = []
with mmap.mmap(tokenised_chu_words_training_file.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_tokens_file:
  word_token_count = 0
  for line in iter(mmap_tokens_file.readline, b""):
    word_offsets.append(word_token_count)
    for token_no in line.decode("utf-8").strip().split(",")[0].split(" "):
      tokens_list.append(int(token_no))
      word_token_count += 1

In [381]:
tokens_tensor = torch.tensor(tokens_list, dtype=torch.float32)

In [382]:
sentence_offsets = []
subtext_offsets = []
text_offsets = []
row_no = 0
sentence_no_prev = 0
subtext_no_prev = 0
text_id_prev = 0
token_count = 0
for row in csv.DictReader(open("../../chu_words_tagged.csv", "r"), delimiter="|"):
    sentence_no = int(row["sentence_no"])
    subtext_no = int(row["subtitle_id"])
    text_id_no = int(row["text_id"])
    if sentence_no != sentence_no_prev:
        sentence_offsets.append(word_offsets[row_no])
        sentence_no_prev = sentence_no
    if text_id_no != text_id_prev:
        text_offsets.append(word_offsets[row_no])
        text_id_prev = text_id_no
        subtext_offsets.append(word_offsets[row_no])
        subtext_no_prev = subtext_no
    elif subtext_no != subtext_no_prev:
        subtext_offsets.append(word_offsets[row_no])
        subtext_no_prev = subtext_no
    row_no += 1

In [383]:
len(tokens_list), len(word_offsets), len(sentence_offsets), len(subtext_offsets), len(text_offsets)

(347761, 242536, 27241, 407, 9)

In [384]:
tokens_dict = {}
tokens_dict_reversed = {}
with mmap.mmap(bpe_token_indices_file.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_bpe_file:
    for bin_line in iter(mmap_bpe_file.readline, b""):
        line = bin_line.decode("utf-8").strip()
        split_line = line.split(",")
        tokens_dict[int(split_line[0])] = split_line[1].strip()
        tokens_dict_reversed[split_line[1]] = int(split_line[0])

In [385]:
retrieveSentence(27239, tokens_list, sentence_offsets)

347760


'сѫтъ же і іна многа ѣже створі іс ѣже аште по едіномоу спана бъіваѭтъ ні самомоу мноѭ вьсемоу міроу въмѣстіті пішемъіхъ кънігъ'

In [386]:
tokens_list[0:10], word_offsets[0:10], sentence_offsets[0:10]

([47, 2264, 1293, 675, 1849, 85, 95, 19, 218, 780],
 [0, 1, 4, 9, 11, 12, 13, 15, 16, 18],
 [0, 11, 36, 62, 110, 116, 179, 188, 208, 218])

In [387]:
sentence_offsets_tensor = torch.tensor(sentence_offsets, dtype=torch.int64)
tensor_snt_lngths = torch.diff(sentence_offsets_tensor)
print("Max sentence length:", tensor_snt_lngths.max())
print("Median sentence length:", tensor_snt_lngths.median())
tensor_snt_lngths[3299] = 0
tensor_snt_lngths[14044] = 0
tensor_snt_lngths[21318] = 0
print(tensor_snt_lngths.max())

Max sentence length: tensor(291)
Median sentence length: tensor(10)
tensor(167)


In [388]:
tensor_snt_lngths = torch.diff(sentence_offsets_tensor)
for i in range(len(tensor_snt_lngths)):
    if tensor_snt_lngths[i] == 167:
        print(i)

15995


In [389]:
tokens_tensor = torch.tensor(tokens_list, dtype=torch.int64)

In [390]:
subtext_offsets[0:2]

[0, 1490]

In [391]:
token_embedder = torch.nn.Embedding(num_embeddings=4539, embedding_dim=256, padding_idx=0)
positional_embedder = torch.nn.Embedding(num_embeddings=64, embedding_dim=256)

In [392]:
position_embeddings = positional_embedder(torch.tensor(range(64)))

In [393]:
for i in range(0, 9, 4):
    print(i)

0
4
8


In [394]:
subtext_windows = []
for i in range(len(subtext_offsets)):
    end_idx = len(tokens_list) if i+1 == len(subtext_offsets) else subtext_offsets[i+1]
    subtext_tokens = tokens_tensor[subtext_offsets[i]:end_idx]
    subtext_token_length = subtext_tokens.size(0)
    # leftover = subtext_token_length
    # window_length = 0
    subtext_chunks = []
    for j in range(0, subtext_token_length, 32):
        window_tokens = subtext_tokens[j:j+64]
        subtext_chunks.append(torch.nn.functional.pad(window_tokens, (0, 64-window_tokens.size(0)), value=0))
    subtext_windows.append(torch.stack(subtext_chunks, dim=0))

In [395]:
subtext_windows[1][1], stringifyTokensTensor(subtext_windows[1][1])

(tensor([ 104,   84, 2963,    7,   37,   94,  127,  126,  199,   39, 1658, 2887,
           39, 4134,   89,  225,  964,   39,  457,   90, 3077,   68, 3293,   75,
           39, 4134,   89,   39,   39, 3741,  174,   60,   39,   39,   73,   28,
          293,  197, 3936,  214, 1071,   68,  484,  557,   61,   89,   39, 2962,
          117,   58,  106, 2349,  267,   39, 1248,  267,  135, 1101,   84, 3618,
          212,   39,   84,   51]),
 'лѣ отъ егюпта прѣнесе і ізгъна ѧзъікъі і насаді ѧ такожде і нъінѣ прізьрі на віноградъ съ і насаді і і оукорені і і оуглѫбі млсть твоѭ на нь ограді і острогомь въходъі і ісходъі его ізбаві отъ снѣга і отъ м')

In [400]:
print(retrieveSubtext(1, tokens_list, subtext_offsets))

мол егда хотѧште віноградъ садіті тъі есі хе віноградъ істінънъіі і оць твоі дѣлатель есть тъі своѧ аплъі лозіе нареклъ есі тъі ілѣ отъ егюпта прѣнесе і ізгъна ѧзъікъі і насаді ѧ такожде і нъінѣ прізьрі на віноградъ съ і насаді і і оукорені і і оуглѫбі млсть твоѭ на нь ограді і острогомь въходъі і ісходъі его ізбаві отъ снѣга і отъ мраза і отъ града носіма боуреѭ вш ѣко мілостівъі чклюбець бъ есі і тебѣ


In [397]:
subtext_windows[405][5], stringifyTokensTensor(subtext_windows[405][5])


(tensor([2834,  773,  481, 2637,   75,  126,   14,  293,   43,   70,   39,  878,
           62, 2161, 1330,  526,  226,  951,  608, 1136,  990, 1915,  526, 2161,
          210,  121,  159,  140,  555,  295,  949,  290,  159, 1342,  358,  144,
          438,   84, 1653,   24, 1157,  488,  252, 3675,   56, 4212,  495,  566,
         1342,  311,  178,  328, 3604,  665, 2023,  328,  144,  107,  201,  427,
          104, 2161, 4086,   16]),
 'цръкъве ідеже вьсі іюдѣі сънемлѫтъ сѧ і таі не глахъ нічесоже чьто мѧ въпрашаеші въпросі слъішавъшѧѧ чьто глахъ імъ се сі вѣдѧтъ еже рѣхъ азъ сі рекъшоу емоу едінъ отъ прѣстоѧштіхъ слоугъ оударі въ ланітѫ іса рекъ тако лі отъвѣштаваеші архіереові отъвѣшта емоу іс аште зълѣ глахъ сьвѣдѣтельствоуі')

In [398]:
transformer_encoder = torch.nn.TransformerEncoderLayer(d_model=256, nhead=8)
print(transformer_encoder)

TransformerEncoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (linear1): Linear(in_features=256, out_features=2048, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (linear2): Linear(in_features=2048, out_features=256, bias=True)
  (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (dropout1): Dropout(p=0.1, inplace=False)
  (dropout2): Dropout(p=0.1, inplace=False)
)
